# M7 — Warm-Start from E7 Reasoning Specialist

Loads E7's LoRA adapter (best reasoning), then fine-tunes for **1 epoch at 1/5 LR** with CW-WSFT + entropy matching.

**Goal:** Preserve E7's strong reasoning (0.83/0.86/0.96 on correctness/completeness/coherence) while fixing its weak accuracy (0.42) and safety (0.74).

**⚠️ Before running:** copy `e7_decision_only_sft/` folder from Drive to `~/kd_project/data/`

In [1]:
# Cell 0: Pick model
import torch
torch.cuda.set_per_process_memory_fraction(0.95)

# TRAIN_MODEL = "qwen25_1p5b_base"
# TRAIN_MODEL = "qwen25_3b_base"
TRAIN_MODEL = "qwen25_7b_base"

print(f"Will train: {TRAIN_MODEL}")

Will train: qwen25_7b_base


In [2]:
# Cell 1: Imports and config
import os, sys, json, math, re
from typing import List, Dict, Any, Tuple
from tqdm import tqdm

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer,
)
from peft import PeftModel

sys.path.insert(0, os.path.expanduser(""))
from shared_utils import (
    RUNS_DIR, MAX_SEQ_LEN, SEED,
    W_FORMAT, W_DECISION, W_EXPL, LAMBDA, STUDENTS,
    load_data,
    get_section_spans, in_any_span,
    compute_alpha, compute_mean_confidence,
    find_decision_span_chars, teacher_section_entropy_mean,
    build_prompt_text, tokenize_and_mask, FlexibleCollator,
)

OUT_DIR = os.path.join(RUNS_DIR, "m7_warmstart_from_e7")
os.makedirs(OUT_DIR, exist_ok=True)

# ⚠️ UPDATE THIS
E7_BASE_DIR = os.path.expanduser("data/e7_decision_only_sft")

WARMSTART_EPOCHS = 1
WARMSTART_LR = 4e-5     # 1/5 of normal LR
W_CW_WSFT = 0.7
W_ENT = 0.3

prompts, teacher_rows = load_data()
MEAN_C = compute_mean_confidence(teacher_rows)

e7_path = os.path.join(E7_BASE_DIR, TRAIN_MODEL)
print(f"E7 adapter: {e7_path} {'✅' if os.path.exists(e7_path) else '❌ NOT FOUND'}")
print(f"Config: {WARMSTART_EPOCHS} epoch, LR={WARMSTART_LR}, loss={W_CW_WSFT}*L_cw + {W_ENT}*L_ent")

c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Loaded 5000 teacher samples from: data\clinical_pharm_teacher_train_6000.jsonl
E7 adapter: data/e7_decision_only_sft\qwen25_7b_base ✅
Config: 1 epoch, LR=4e-05, loss=0.7*L_cw + 0.3*L_ent


In [3]:
# Cell 2: Precomputed Dataset
class M7DatasetFast(Dataset):
    def __init__(self, rows, prompts, tokenizer, is_instruct, mean_c):
        print("  Precomputing dataset...")
        self.items = []
        for idx in tqdm(range(len(rows)), desc="  Tokenizing"):
            r = rows[idx]
            prompt = prompts[r["id"]]
            answer = r["teacher_text"]
            prompt_text = build_prompt_text(tokenizer, prompt, is_instruct)
            input_ids, offsets, labels, answer_start = tokenize_and_mask(tokenizer, prompt_text, answer)

            wsft_weights = [0.0] * len(input_ids)
            spans = get_section_spans(answer)
            dec_spans = [(answer_start + s, answer_start + e) for s, e in spans["decision"]]
            expl_spans = [(answer_start + s, answer_start + e) for s, e in spans["explanation"]]
            for i, (s, e) in enumerate(offsets):
                if e <= s: continue
                if s >= answer_start:
                    w = W_FORMAT
                    if in_any_span(s, e, dec_spans): w = W_DECISION
                    if in_any_span(s, e, expl_spans): w = W_EXPL
                    wsft_weights[i] = float(w)
            active_w = [w for w in wsft_weights if w > 0]
            if active_w:
                mean_w = sum(active_w) / len(active_w)
                if mean_w > 1e-6: wsft_weights = [w / mean_w if w > 0 else 0.0 for w in wsft_weights]

            alpha = compute_alpha(r, mean_c)
            dec_mask = torch.zeros(len(input_ids), dtype=torch.bool)
            dec_span_chars = find_decision_span_chars(answer)
            if dec_span_chars:
                ds, de = dec_span_chars
                full_ds, full_de = answer_start + ds, answer_start + de
                for i, (s, e) in enumerate(offsets):
                    if labels[i] != -100 and not (e <= full_ds or s >= full_de):
                        dec_mask[i] = True
            ent_teacher = teacher_section_entropy_mean(r, dec_span_chars)

            self.items.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "wsft_weights": torch.tensor(wsft_weights, dtype=torch.float),
                "alpha": torch.tensor(alpha, dtype=torch.float),
                "dec_mask": dec_mask,
                "ent_teacher": ent_teacher.float(),
            })
        print(f"  ✅ Precomputed {len(self.items)} samples")

    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

print("✅ Dataset ready")

✅ Dataset ready


In [4]:
# Cell 3: Trainer — CW-WSFT + entropy matching
class M7Trainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._ce_loss = torch.nn.CrossEntropyLoss(reduction="none")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        wsft_weights = inputs.pop("wsft_weights")
        alpha = inputs.pop("alpha")
        dec_mask = inputs.pop("dec_mask")
        ent_teacher = inputs.pop("ent_teacher")

        outputs = model(**inputs)
        logits = outputs.logits
        sl = logits[:, :-1, :].contiguous()
        ll = labels[:, 1:].contiguous()
        sw = wsft_weights[:, 1:].contiguous()
        dm = dec_mask[:, 1:]
        vocab, B = sl.size(-1), sl.size(0)

        tl = self._ce_loss(sl.view(-1, vocab), ll.view(-1)).view(ll.size())
        active = (ll != -100).float()

        # L_cw_wsft
        w = sw * active
        denom = active.sum(dim=1).clamp(min=1.0)
        per_sample = (tl * w).sum(dim=1) / denom
        L_cw = (per_sample * alpha).mean()

        # L_dec_ent — memory efficient
        dec_active_mask = dm & (ll != -100)
        ent_student = torch.zeros(B, device=logits.device)
        for b in range(B):
            idx = dec_active_mask[b].nonzero(as_tuple=True)[0]
            if len(idx) == 0: continue
            local_logits = sl[b, idx, :]
            local_probs = torch.softmax(local_logits, dim=-1)
            local_ent = -(local_probs * torch.log(local_probs + 1e-9)).sum(-1)
            ent_student[b] = local_ent.mean()
        L_ent = LAMBDA * ((ent_student - ent_teacher.to(logits.device)) ** 2).mean()

        loss = W_CW_WSFT * L_cw + W_ENT * L_ent
        return (loss, outputs) if return_outputs else loss

print("✅ Trainer ready")

✅ Trainer ready


In [ ]:
# Cell 4: Load E7 adapter and continue training
cfg = STUDENTS[TRAIN_MODEL]
print(f"\n{'='*50} M7: {TRAIN_MODEL} {'='*50}")

out_path = os.path.join(OUT_DIR, TRAIN_MODEL)
os.makedirs(out_path, exist_ok=True)

e7_path = os.path.join(E7_BASE_DIR, TRAIN_MODEL)
assert os.path.exists(e7_path), f"❌ E7 checkpoint not found: {e7_path}"

print("  Loading base model...")
tokenizer = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    cfg["path"], torch_dtype=torch.bfloat16,
    device_map="auto", trust_remote_code=True,
)

print(f"  Loading E7 LoRA from {e7_path}...")
model = PeftModel.from_pretrained(base_model, e7_path, is_trainable=True)
model.train()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  ✅ E7 adapter loaded: {trainable:,} trainable params")

dataset = M7DatasetFast(teacher_rows, prompts, tokenizer, cfg["is_instruct"], MEAN_C)
collator = FlexibleCollator(tokenizer, extra_1d_fields=["wsft_weights", "dec_mask"],
                            extra_scalar_fields=["alpha", "ent_teacher"])

if "1p5b" in TRAIN_MODEL:
    bs, ga = 4, 8
elif "3b" in TRAIN_MODEL:
    bs, ga = 2, 16
else:
    bs, ga = 1, 32

trainer = M7Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=out_path, num_train_epochs=WARMSTART_EPOCHS,
        per_device_train_batch_size=bs, gradient_accumulation_steps=ga,
        learning_rate=WARMSTART_LR,
        logging_steps=25, save_strategy="no",
        bf16=True, seed=SEED, report_to="none",
        remove_unused_columns=False, dataloader_num_workers=0,
    ),
    train_dataset=dataset, data_collator=collator,
)
trainer.train()
model.save_pretrained(out_path)
tokenizer.save_pretrained(out_path)
print(f"\n✅ {TRAIN_MODEL} saved → {out_path}")
print("⚠️ To train next model: change Cell 0, RESTART KERNEL, run all cells")


================================================== M7: qwen25_7b_base ==================================================
  Loading base model...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:09<00:00, 34.37it/s]


  Loading E7 LoRA from data/e7_decision_only_sft\qwen25_7b_base...
  ✅ E7 adapter loaded: 10,092,544 trainable params
  Precomputing dataset...


  Tokenizing: 100%|██████████| 5000/5000 [00:07<00:00, 680.71it/s]


  ✅ Precomputed 5000 samples


Step,Training Loss
25,16.981532
50,16.428967
75,16.566667
